In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy.stats import t
from pathlib import Path
from joblib import Parallel, delayed
import json

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'mining-area').mkdir(parents=True, exist_ok=True)
import sys
sys.path.insert(0, str(ROOT / '3-mining-area-simulation'))


# Estimate mining-area expansion (2050)

Input allocated extraction carries the complete D/R/A tag, because the area relationship determines effective capacity before allocation. At 1 km, retain the original suitability × geological-reserve ranking and fill each cell only up to min(reserve, extraction at 1 km²). Transfer the remaining volume to the next ranked cell. Sum the resulting bounded sub-cell footprints to 50 km. Geological reserves and national extraction totals are unchanged.

Inputs: `allocated-extraction-2023-2050-{scenario}-B{beta}-{res}-D{Ld}-R{Lr}-A{La}.tif`, original 1 km reserves and suitability, fitted extraction–area relationship.

Uncertainty ranges use the observed minimum and maximum across all 27 D/R/A combinations.


In [ ]:
# shared 1 km layers (grid + suitability); reused across all scenario × beta runs
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
country_raster_50km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif')
mining_suitability_1km = rio.open(RESULTS / 'global-mining-suitability-1km.tif').read(1)

In [ ]:
def func_1(x, a, b):
    return a * x ** b

In [ ]:
# empirical extraction (Mt) -> disturbed area (ha) power law (same data as notebook 3)
df_rel = pd.read_csv(RESULTS / 'extraction-area-relationship.csv')
X_rel = df_rel['E (Mt)'].values
Y_rel = df_rel['A (ha)'].values
popt_1, pcov_1 = curve_fit(func_1, X_rel, Y_rel)
a_1, b_1 = popt_1
dof_1 = len(X_rel) - len(popt_1)
print(f'A = {a_1:.3f} * E^{b_1:.3f}  (n={len(X_rel)})')

In [ ]:
def predict_interval(X, popt, pcov, dof):
    """95% fitted power-law interval, with a physically non-negative area lower bound."""
    a, b = popt
    cov_aa, cov_ab, cov_bb = pcov[0, 0], pcov[0, 1], pcov[1, 1]
    mask_valid = (X > 0)
    X_safe = np.where(mask_valid, X, 1e-12)
    y_best = a * X_safe ** b
    dy_da = X_safe ** b
    dy_db = a * X_safe ** b * np.log(X_safe)
    sigma_y2 = cov_aa * dy_da ** 2 + 2 * cov_ab * dy_da * dy_db + cov_bb * dy_db ** 2
    sigma_y = np.sqrt(sigma_y2)
    tval = t.ppf(0.975, dof)
    # Apply the physical area floor to each 1 km contribution BEFORE aggregation.
    lower = np.where(mask_valid, np.maximum(y_best - tval * sigma_y, 0), 0)
    upper = np.where(mask_valid, y_best + tval * sigma_y, 0)
    return lower.astype(np.float32), upper.astype(np.float32)

In [ ]:
def estimate_mining_area(scenario, beta, demand_level='M', reserve_level='M', area_level='M', res='50km',
                         suit_1km=None, cap_1km=None, fit=None, output_dir=None):
    """Allocate to original suitability × reserve ranking under the 1 km² cap.

    Both input allocation and output footprint carry D/R/A tags. The area cap
    enters coarse allocation first; volume is conserved at fine redistribution.
    """
    from area_capacity import area_km2, extraction_limit, assign_extraction
    if suit_1km is None:
        suit_1km = rio.open(RESULTS / 'global-mining-suitability-1km.tif').read(1)
    if cap_1km is None:
        cap_1km = rio.open(RESULTS / 'mining-area' / f'aggregate-reserve-1km-{reserve_level}.tif').read(1)
    a, b, popt, pcov, dof = fit if fit is not None else (a_1, b_1, popt_1, pcov_1, dof_1)

    limit = extraction_limit(popt, pcov, dof, area_level)
    diagnostics = dict(max_subcell_area_km2=0., max_mass_error_Mt=0., occupied_subcells=0)

    # 50 km -> 1 km mapping: scale + origin offset are constants -> compute ONCE (not per cell)
    src_ref = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    dst_ref = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
    scale = {'10km': 10, '50km': 50}[res]
    off_i = int(abs(dst_ref.transform[5] - src_ref.transform[5]) / 1e3)
    off_j = int(abs(dst_ref.transform[2] - src_ref.transform[2]) / 1e3)

    alloc_tag = f'{scenario}-B{beta:.2f}-{res}-D{demand_level}-R{reserve_level}-A{area_level}'
    out_tag = alloc_tag
    allocated_extraction = rio.open(
        RESULTS / 'extraction-rasters' / f'allocated-extraction-2023-2050-{alloc_tag}.tif').read(1)

    mining_area_expansion = np.zeros(src_ref.shape, dtype=np.float64)
    i_set, j_set = np.where(allocated_extraction > 0)
    for i, j in zip(i_set, j_set):
        i_1, i_2 = i * scale - off_i, (i + 1) * scale - off_i
        j_1, j_2 = j * scale - off_j, (j + 1) * scale - off_j
        suitability_slice = suit_1km[i_1:i_2, j_1:j_2] / 1e4
        avail_slice = cap_1km[i_1:i_2, j_1:j_2]
        Q_total = allocated_extraction[i, j]

        score_flat = (suitability_slice * avail_slice).flatten()
        avail_flat = avail_slice.flatten()
        m = avail_flat > 0
        score_flat, avail_flat = score_flat[m], avail_flat[m]
        if score_flat.size == 0:
            raise ValueError('Extraction assigned to a cell without capacity')

        order = np.argsort(score_flat)[::-1]            # best (most suitable, highest reserve) first
        q_max = np.minimum(avail_flat[order].astype(np.float64), limit)
        q_assigned = assign_extraction(Q_total, q_max)
        area_each = area_km2(q_assigned, popt, pcov, dof, area_level)
        if area_each.max(initial=0) > 1 + 1e-9:
            raise ValueError(f'1 km sub-cell area cap exceeded: {out_tag}, {i}, {j}')
        diagnostics['max_subcell_area_km2'] = max(diagnostics['max_subcell_area_km2'], float(area_each.max(initial=0)))
        diagnostics['max_mass_error_Mt'] = max(diagnostics['max_mass_error_Mt'], float(abs(q_assigned.sum()-Q_total)))
        diagnostics['occupied_subcells'] += int(np.count_nonzero(q_assigned))
        mining_area_expansion[i, j] = float(np.sum(area_each))

    if not np.isfinite(mining_area_expansion).all() or (mining_area_expansion < 0).any():
        raise ValueError(f'Invalid mining-area values for {out_tag}: expected finite, non-negative areas')

    total_expansion  = float(np.sum(mining_area_expansion))
    total_extraction = float(np.sum(allocated_extraction)) / 1e3

    output_dir = Path(output_dir) if output_dir is not None else RESULTS / 'mining-area'
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f'mining-area-expansion-2050-{out_tag}.tif'
    with rio.open(out_path, 'w', driver='GTiff',
        height=src_ref.shape[0], width=src_ref.shape[1], count=1,
        dtype=mining_area_expansion.dtype, crs=src_ref.crs,
        transform=src_ref.transform, compress='lzw') as dst:
        dst.write(mining_area_expansion, 1)
        if hasattr(dst, 'update_tags'):
            dst.update_tags(**diagnostics, area_cap_km2='1', extraction_limit_Mt=str(limit))
    return f'{out_tag}: extraction {total_extraction:.1f} Gt -> footprint {total_expansion:,.0f} km^2'

## Run — eight strategies at β = 0, with all 27 D/R/A combinations


In [ ]:
scenario_set_1 = ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']
LEVELS = ['L', 'M', 'H']
N_JOBS = 10                                          # processes; raise toward os.cpu_count()-2 if RAM allows

# Self-contained shared inputs (does NOT depend on the cross / bounds cells below):
#   suitability + power-law fit come from the early cells; cap_1km MUST match the reserve level,
#   so preload all three reserve rasters once and pick per-task (joblib memmaps them to workers).
suit_1km       = mining_suitability_1km
fit            = (a_1, b_1, popt_1, pcov_1, dof_1)
cap_by_reserve = {lvl: rio.open(RESULTS / 'mining-area' / f'aggregate-reserve-1km-{lvl}.tif').read(1)
                  for lvl in LEVELS}

# Full factorial: 8 scenarios x {D,R,A} in {L,M,H}^3 = 8 x 27 = 216 footprints, all at beta = 0.
combos = [(scen, dl, rl, al)
          for scen in scenario_set_1
          for dl in LEVELS for rl in LEVELS for al in LEVELS]

msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(scen, 0.0, demand_level=dl, reserve_level=rl, area_level=al,
                                  suit_1km=suit_1km, cap_1km=cap_by_reserve[rl], fit=fit)
    for scen, dl, rl, al in combos
)
for msg in msgs:
    print(' ', msg)

## Run — scenario × β sweep with central D/R/A


In [ ]:
_cfg = json.load(open(RESULTS / 'sweep-config.json'))   # sweep grid defined once in 5-2-map-extraction
SCENARIOS, BETAS = _cfg['SCENARIOS'], _cfg['BETAS']
N_JOBS    = 10                                                   # processes; raise toward (os.cpu_count()-2) if RAM allows

# Load the shared read-only 1 km layers ONCE. Passing them as args lets joblib memmap them to the
# workers (dumped once, mmap'd read-only) instead of each of the 50-60 tasks re-reading from disk
# or each pickling a full copy -> low memory, no per-task I/O.
suit_1km = mining_suitability_1km                              # already loaded in the shared-layers cell
cap_1km  = rio.open(RESULTS / 'mining-area' / 'aggregate-reserve-1km-M.tif').read(1)   # reserve_level='M'
fit      = (a_1, b_1, popt_1, pcov_1, dof_1)

combos = [(scen, b) for scen in SCENARIOS for b in BETAS]
msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(scen, b, demand_level='M', reserve_level='M', area_level='M',
                                  suit_1km=suit_1km, cap_1km=cap_1km, fit=fit)
    for scen, b in combos
)
for msg in msgs:
    print(' ', msg)

## Full uncertainty factorial at CON and IAP (β = 0.7)

Evaluate all 27 combinations and obtain bounds from their results.


In [ ]:
SIGNPOSTS = [('BAU', 0.7), ('CE', 0.7)]
bound_combos = [(s, b, d, r, a) for s, b in SIGNPOSTS
                for d in LEVELS for r in LEVELS for a in LEVELS]
bound_msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(s, b, demand_level=d, reserve_level=r, area_level=a,
                                  suit_1km=suit_1km, cap_1km=cap_by_reserve[r], fit=fit)
    for s, b, d, r, a in bound_combos)
for message in bound_msgs:
    print(message)
